# 第6章 運転シーンを読み解く (危険予測と運転判断)

Cosmos 3 は自動運転のデータでも学習されていて、車載カメラの映像について「何に注意すべきか」「次にどうするか」を答えられます。

1. 自動運転の計画役として、次の行動を答えさせる
2. 安全のために重要な物を挙げさせる (考えるモード)
3. この先数秒の危険を予測させ、その物を枠で示させる
4. 自車が進むべき道筋を画像の座標で答えさせる
5. 判断を JSON で出させる (ほかのプログラムにつなぐ形)
6. 動画が物理的にありえるかを判定させる (生成した動画のチェックに使える)
7. 自分のドライブレコーダーの映像で試す

**注意**: ここでの答えはモデルの推測です。実際の運転の判断に使えるほどの信頼性はありません (モデルカードにも安全が重要な用途にはそのまま使わないよう書かれています)。

## 準備

**ランタイム**: 「ランタイム → ランタイムのタイプを変更」で **L4** か **A100** の GPU を選んでください (T4 は対象外。Ampere 以降の GPU と bf16 が前提です)。
Reasoner (文章で答える部分) だけを使うので、GPU メモリは 10GB 前後で足ります。

上から順に実行してください。「モデルを読み込む」セルは初回に 5GB ほどダウンロードします。

In [ ]:
# @title GPU を確かめる
import torch
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
    assert torch.cuda.get_device_capability()[0] >= 8, "Ampere 以降の GPU (L4 / A100 など) を選んでください。T4 では動きません。"
else:
    print("GPU が見つかりません。Colab ならランタイムのタイプで L4 か A100 を選んでください。")

In [ ]:
# @title インストール (1〜2 分)
!pip install -q -U "transformers>=5.18" accelerate av huggingface_hub

In [ ]:
# @title サンプル素材を取ってくる関数
import os
import urllib.request
from pathlib import Path

ASSETS = Path(os.environ.get("COSMOS_ASSETS", "assets"))
ASSETS.mkdir(parents=True, exist_ok=True)
# NVIDIA の公式クックブック (cosmos-framework) とモデルカードに置かれているお手本の素材
FW = "https://media.githubusercontent.com/media/NVIDIA/cosmos-framework/main/.agents/skills/cookbooks/cosmos3"
HF = "https://huggingface.co/nvidia/Cosmos3-Edge/resolve/main/assets"


def fetch(url):
    """URL の素材を assets/ に落として、そのパスを返す (2 回目からは落とさない)"""
    path = ASSETS / url.rsplit("/", 1)[-1]
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return str(path)

In [ ]:
# @title モデルを読み込む (初回は 5GB ほどダウンロード)
import json
import re

from IPython.display import Image as IPImage, Video, display
from PIL import Image, ImageDraw, ImageOps
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "nvidia/Cosmos3-Edge"
# Transformers は統合チェックポイントから Reasoner の部分だけを読み込む (生成用の部分は捨てる)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [ ]:
# @title 質問用の関数を用意する
import json
import re

import av
import numpy as np
from PIL import Image, ImageDraw, ImageOps
from transformers.video_utils import VideoMetadata

VIDEO_EXT = (".mp4", ".mov", ".webm", ".avi", ".mkv")


def load_image(path, max_side=1280):
    """画像を RGB で読む。大きすぎる画像はトークンが増えて遅くなるので縮める (座標は 0〜1000 の比率なので影響しない)"""
    img = path if isinstance(path, Image.Image) else ImageOps.exif_transpose(Image.open(path))
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side))
    return img


def read_video(path, fps=4, max_side=640, max_frames=256):
    """動画を fps 枚/秒に間引いて読む。返すのはコマの配列と、各コマの時刻を伝えるための情報 (VideoMetadata)。
    Reasoner は各コマに「<1.5 seconds>」のような時刻を付けて読むので、時刻を答える問題 (第4章) にはこの情報が欠かせない"""
    frames, t_next = [], 0.0
    with av.open(path) as c:
        for f in c.decode(video=0):
            if f.time is None or f.time + 1e-6 < t_next:
                continue
            img = f.to_image()
            img.thumbnail((max_side, max_side))  # 1 コマのトークン数を抑える
            frames.append(np.asarray(img.convert("RGB")))
            t_next += 1 / fps
            if len(frames) >= max_frames:
                break
    n = len(frames)
    return np.stack(frames), VideoMetadata(total_num_frames=n, fps=fps, frames_indices=list(range(n)), duration=n / fps)


def ask(prompt, media=None, thinking=False, max_new_tokens=2048, fps=4, seed=0):
    """画像・動画 (なくてもよい) と質問を渡して、答えの文字列を返す。

    thinking=True のときは考えた過程 (<think>…</think>) を出してから答える。考えた過程は ask.last_think に残る。
    動画は fps 枚/秒で間引いて読む (推奨は 4)。
    """
    media = [] if media is None else media if isinstance(media, (list, tuple)) else [media]
    content, metadata = [], []
    for m in media:  # 画像・動画は質問文より前に置く (公式の推奨)
        if isinstance(m, str) and m.lower().endswith(VIDEO_EXT):
            frames, meta = read_video(m, fps=fps)
            content.append({"type": "video", "video": frames})
            metadata.append(meta)
        else:
            content.append({"type": "image", "image": load_image(m)})
    content.append({"type": "text", "text": prompt})
    # 動画は read_video で間引き済みなので、ここではもう間引かない
    video_kwargs = {"video_metadata": metadata, "do_sample_frames": False} if metadata else {}
    inputs = processor.apply_chat_template(
        [{"role": "user", "content": content}],
        tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
        enable_thinking=thinking, **video_kwargs,
    ).to(model.device)
    # 公式のプロンプトガイドのサンプリング設定 (考える / 考えない で値が違う)
    sampling = dict(temperature=0.6, top_p=0.95, top_k=20) if thinking else dict(temperature=0.7, top_p=0.8, top_k=20)
    torch.manual_seed(seed)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, **sampling)
    text = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    ask.last_think = ""
    if "</think>" in text:
        think, text = text.split("</think>", 1)
        ask.last_think = think.replace("<think>", "").strip()
    return text.strip()


def parse_json(text):
    """答えの中の JSON (```json … ``` で囲まれていてもよい) を取り出す。読めなければ None。
    小さなモデルの答えは、最後に余計な括弧が付く・真偽値の後ろに " が付く・最後に余計なカンマがある、
    といった崩れ方をするので、括弧の対応を数えて最初のまとまりだけを取り出し、よくある崩れを直してから読む"""
    m = re.search(r"```(?:json)?\s*(.*?)```", text, re.S)
    body = m.group(1) if m else text
    starts = [i for i in (body.find("["), body.find("{")) if i >= 0]
    if not starts:
        return None
    start = min(starts)
    depth, in_str, esc, end = 0, False, False, len(body)
    for k in range(start, len(body)):
        ch = body[k]
        if in_str:
            esc = (ch == "\\") and not esc
            if ch == '"' and not esc:
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch in "[{":
            depth += 1
        elif ch in "]}":
            depth -= 1
            if depth == 0:
                end = k + 1
                break
    chunk = body[start:end]
    fixed = re.sub(r'\b(true|false|null)"', r"\1", re.sub(r",\s*([\]}])", r"\1", chunk))
    for attempt in (chunk, fixed):
        try:
            return json.loads(attempt)
        except json.JSONDecodeError:
            pass
    print("(JSON として読めませんでした)")
    return None


def load_reasoner():
    """Reasoner を読み込む (生成の章で、Generator と入れ替えて使うとき用)"""
    global model, processor
    from transformers import AutoModelForImageTextToText, AutoProcessor
    model = AutoModelForImageTextToText.from_pretrained("nvidia/Cosmos3-Edge", dtype=torch.bfloat16, device_map="auto")
    processor = AutoProcessor.from_pretrained("nvidia/Cosmos3-Edge")

In [ ]:
# @title 座標を画像に描く関数
from PIL import ImageFont

COLORS = ["#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4", "#42d4f4", "#f032e6", "#bfef45"]


def to_px(x, y, img):
    """Cosmos の座標 (左上が原点、x・y それぞれ 0〜1000 に正規化) をピクセルに直す"""
    return x / 1000 * img.width, y / 1000 * img.height


def draw(img, items, path=False, width=3):
    """bbox_2d (枠) と point_2d (点) を描いた画像を返す。path=True なら点を順に線でつなぐ (軌跡)"""
    img = load_image(img).copy()
    d = ImageDraw.Draw(img)
    font = ImageFont.load_default(size=max(14, img.width // 45))
    items = items if isinstance(items, list) else [items]
    pts = []
    for i, it in enumerate(items):
        c = COLORS[i % len(COLORS)]
        label = str(it.get("label", i))
        if "bbox_2d" in it:
            x1, y1 = to_px(*it["bbox_2d"][:2], img)
            x2, y2 = to_px(*it["bbox_2d"][2:4], img)
            d.rectangle([x1, y1, x2, y2], outline=c, width=width)
            d.text((x1 + 4, y1 + 2), label, fill=c, font=font, stroke_width=2, stroke_fill="white")
        if "point_2d" in it:
            x, y = to_px(*it["point_2d"][:2], img)
            pts.append((x, y, label))
    if path and len(pts) > 1:
        d.line([(x, y) for x, y, _ in pts], fill="#ffd400", width=width + 1)
    r = max(5, img.width // 110)
    for j, (x, y, label) in enumerate(pts):
        c = "#ffd400" if path else COLORS[j % len(COLORS)]
        d.ellipse([x - r, y - r, x + r, y + r], fill=c, outline="black", width=2)
        text = str(j + 1) if path else label
        d.text((x + r + 3, y - r), text, fill=c, font=font, stroke_width=2, stroke_fill="black")
    return img

In [ ]:
# @title 動画から 1 コマ取り出す関数
import av


def frame_at(path, t):
    with av.open(path) as c:
        s = c.streams.video[0]
        c.seek(int(max(t, 0) / s.time_base), stream=s)
        last = None
        for f in c.decode(s):
            last = f
            if f.time is not None and f.time >= t:
                break
        return last.to_image()

## 1. 次の行動を答えさせる

公式のプロンプトガイドで使われている、自動運転の計画役としての聞き方です。答えは一人称 (「私は…」) で返ってきます。

In [ ]:
video = fetch(f"{FW}/reasoner/assets/drive_scene_next_action.mp4")
display(Video(video, embed=True, width=480))
print(ask("You are an autonomous vehicle planning system. The video depicts the observation from the vehicle's camera. "
          "You need to observe the critical objects in the environment and reason your next action and the driving trajectory ahead.", video))

## 2. 安全のために重要な物を挙げさせる

In [ ]:
video = fetch(f"{FW}/reasoner/assets/action_cot_driving_scene.mp4")
display(Video(video, embed=True, width=480))
answer = ask("The video depicts the observation from the vehicle's camera. You need to think step by step and identify "
             "the objects in the scene that are critical for safe navigation.", video, thinking=True)
print("--- 考えた過程 ---\n", ask.last_think[:1500])
print("\n--- 答え ---\n", answer)

## 3. この先の危険を予測させ、枠で示させる

まず動画全体から「この先数秒で危なくなりそうな物」を答えさせ、次に最後のコマでその物を枠で囲ませます。
「転がってきたボールの後ろから子どもが出てくるかもしれない」のような、見えていないことの予測が世界モデルらしい使い方です。

In [ ]:
video = fetch(f"{HF}/example_action_id_av_1_input.mp4")
display(Video(video, embed=True, width=480))
hazard = ask("You are driving this car. What could become a hazard in the next few seconds, and why? "
             "Answer in 3 short bullet points.", video, thinking=True)
print(hazard)

last = frame_at(video, 5.9)
answer = ask(f"""Possible hazards:
{hazard}
Locate the road users and objects mentioned above in this image.
Return a JSON list like this: [{{"bbox_2d": [x1, y1, x2, y2], "label": "name"}}]""", last)
display(draw(last, parse_json(answer) or []))

## 4. 自車が進むべき道筋を座標で答えさせる

第5章の「手先の道筋」と同じ形 (`point_2d` のリスト) と同じ聞き方 (考える書式を質問文に書く) で、自車が通るべき道筋を画像の上で答えさせます。
第8章では、これとは逆に「道筋 (行動) を渡して、未来の映像を作らせる」ことをします。

In [ ]:
first = load_image(fetch(f"{FW}/generator/action/assets/images/av_0.jpg"))  # 第8章でも使う昼の道路の 1 コマ
answer = ask('You are driving this car. Specify the 2D trajectory the ego vehicle should follow in the next 5 seconds in pixel space. '
             'Return the trajectory coordinates in JSON format like this: {"point_2d": [x, y], "label": "ego trajectory"}.\n'
             'Answer the question using the following format:\n\n<think>\nYour reasoning.\n</think>\n\n'
             'Write your final answer immediately after the </think> tag.', first)
display(draw(first, parse_json(answer) or [], path=True))

## 5. 判断を JSON で出させる

In [ ]:
answer = ask("""You are an autonomous vehicle planning system. Watch the video from the vehicle's camera and decide what to do next.
Return one JSON object with these keys:
"decision": one of "keep_lane", "slow_down", "stop", "turn_left", "turn_right", "change_lane_left", "change_lane_right",
"target_speed_kmh": a number,
"critical_objects": a list of short names,
"reason": one sentence.""", video)
decision = parse_json(answer)
print(json.dumps(decision, indent=2, ensure_ascii=False) if decision else answer)

## 6. 動画が物理的にありえるかを判定させる

公式のお手本の、物理的にありえるか (物が消えない・形が保たれる・動きが連続しているか) を選ばせる問題です。
第7章以降で Cosmos 自身が作った動画を、この聞き方でチェックできます (第12章で使います)。

In [ ]:
video = fetch(f"{FW}/reasoner/assets/physical_plausibility.mp4")
display(Video(video, embed=True, width=480))
print(ask("""Is this video physically plausible/possible according to your understanding of e.g. object permanence, shape constancy (objects maintain shape over time), continuous trajectories of objects? Assume it is the normal laws of physics.
Your answer should be based on the events in the video and ignore the quality of the simulation engine. The rising wall is part of the experiment setup and should not be judged for plausibility.
(A) Possible
(B) Impossible""", video))

## 7. 自分のドライブレコーダーの映像で試す

10 秒程度の mp4 がおすすめです (長いと遅くなります)。

In [ ]:
# @title 自分の動画で試す (colab-only)
from google.colab import files

up = files.upload()
path = next(iter(up))
display(Video(path, embed=True, width=480))
answer = ask("You are an autonomous vehicle planning system. The video depicts the observation from the vehicle's camera. "
             "You need to observe the critical objects in the environment and reason your next action and the driving trajectory ahead.", path)
print(answer)

## うまくいかないとき

- **日付や速度の表示に引っ張られる**: ドライブレコーダーの文字が映っている部分は切り落としてから入れると、場面の説明が安定します
- **夜間・雨で物を見落とす**: Edge は小さなモデルなので、暗い場面では見落としが増えます。`thinking=True` にすると改善することがあります
- **答えが毎回違う**: `seed` を変えて何回か聞き、多数決を取るのが手軽です